# ReAct Pattern

**Module:** 10-agentic-ai-concepts

**Notebook:** `03-react-pattern.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **ReAct Overview** with clear contracts and failure modes
- Explain and apply **Trace Format** with clear contracts and failure modes
- Explain and apply **Implementing the Loop** with clear contracts and failure modes
- Explain and apply **Native Tool Calling vs Text ReAct** with clear contracts and failure modes
- Explain and apply **Errors & Recovery** with clear contracts and failure modes
- Explain and apply **Stop Conditions** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — ReAct Pattern

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **ReAct Overview**
2. **Trace Format**
3. **Implementing the Loop**
4. **Native Tool Calling vs Text ReAct**
5. **Errors & Recovery**
6. **Stop Conditions**

Read top-to-bottom once, then revisit weak spots with the exercises.


## ReAct Overview

### Definition
**ReAct** interleaves reasoning with tool actions and observations until a final answer.

### Why it matters
Thinking without acting cannot access live systems; acting without thinking is reckless.

### How it works
Thought → Action → Observation loop with max steps and validated arguments.

### Intuition
Air-traffic control: think, act, read instruments, repeat.

### Pitfalls
- Format drift
- Tool thrash
- Hallucinated observations

### When to use
Tool-using agents with external state.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does ReAct Overview improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "ReAct Overview" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "ReAct Overview"
    notebook: str = "03-react-pattern"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


In [ ]:
# Demo: decision table for applying "ReAct Overview"
options = [
    {"option": "baseline_simple", "quality": 0.7, "cost": 1, "ops": 0.9},
    {"option": "advanced_react_overvi", "quality": 0.85, "cost": 3, "ops": 0.6},
]
for o in options:
    o["utility"] = round(o["quality"] * 2 - 0.3*o["cost"] + 0.5*o["ops"], 3)
best = max(options, key=lambda x: x["utility"])
print("ranked:", sorted(options, key=lambda x: -x["utility"]))
print("prefer:", best["option"])


## Trace Format

### Definition
**Trace Format** is a core building block in 03-react-pattern within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Trace Format typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Trace Format: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Trace Format as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Trace Format as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Trace Format
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Trace Format when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Trace Format" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Trace Format"
    notebook: str = "03-react-pattern"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Trace Format"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Trace Format"}
strong = {"definition": "Trace Format", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Trace Format"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Trace Format", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Trace Format

**Situation:** A team wants to productionize a feature involving **Trace Format**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Implementing the Loop

### Definition
**Implementing the Loop** is a core building block in 03-react-pattern within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Implementing the Loop typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Implementing the Loop: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Implementing the Loop as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Implementing the Loop as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Implementing the Loop
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Implementing the Loop when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Implementing the Loop" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Implementing the Loop"
    notebook: str = "03-react-pattern"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Implementing the Loop"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Implementing the Loop"}
strong = {"definition": "Implementing the Loop", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Implementing the Loop"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Implementing the Loop", "passed": len(checks)-len(failed), "failed": failed})


## Native Tool Calling vs Text ReAct

### Definition
**Native Tool Calling vs Text ReAct** helps you choose among alternatives using explicit criteria rather than hype.

### Why it matters
In agentic systems, weak designs around Native Tool Calling vs Text ReAct typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
List options, define criteria (quality, cost, latency, ops, lock-in), score with evidence, document the decision.

### Intuition
Explain Native Tool Calling vs Text ReAct as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Native Tool Calling vs Text ReAct as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Native Tool Calling vs Text ReAct
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Native Tool Calling vs Text ReAct when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Native Tool Calling vs Text ReAct" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Native Tool Calling vs Text ReAct"
    notebook: str = "03-react-pattern"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


### Worked scenario — Native Tool Calling vs Text ReAct

**Situation:** A team wants to productionize a feature involving **Native Tool Calling vs Text ReAct**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Errors & Recovery

### Definition
**Errors & Recovery** is a core building block in 03-react-pattern within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Errors & Recovery typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Errors & Recovery: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Errors & Recovery as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Errors & Recovery as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Errors & Recovery
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Errors & Recovery when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Errors & Recovery" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Errors & Recovery"
    notebook: str = "03-react-pattern"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Errors & Recovery"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Errors & Recovery"}
strong = {"definition": "Errors & Recovery", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Errors & Recovery"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Errors & Recovery", "passed": len(checks)-len(failed), "failed": failed})


## Stop Conditions

### Definition
**Stop Conditions** is a core building block in 03-react-pattern within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Stop Conditions typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Stop Conditions: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Stop Conditions as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Stop Conditions as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Stop Conditions
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Stop Conditions when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Stop Conditions" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Stop Conditions"
    notebook: str = "03-react-pattern"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Stop Conditions"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Stop Conditions"}
strong = {"definition": "Stop Conditions", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Stop Conditions"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Stop Conditions", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Stop Conditions

**Situation:** A team wants to productionize a feature involving **Stop Conditions**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Comparison Snapshot

Use this table when reviewing designs in **ReAct Pattern**.

| Topic | Do | Don't |
|-------|----|-------|
| ReAct Overview | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Trace Format | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Implementing the Loop | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Native Tool Calling vs Text ReAct | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Errors & Recovery | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Stop Conditions | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| ReAct Overview | Key concept covered in this notebook; see its section for definition and pitfalls |
| Trace Format | Key concept covered in this notebook; see its section for definition and pitfalls |
| Implementing the Loop | Key concept covered in this notebook; see its section for definition and pitfalls |
| Native Tool Calling vs Text ReAct | Key concept covered in this notebook; see its section for definition and pitfalls |
| Errors & Recovery | Key concept covered in this notebook; see its section for definition and pitfalls |
| Stop Conditions | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **ReAct Pattern** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **10-agentic-ai-concepts**.


## Try It Yourself

1. Implement a failing test/fixture for **ReAct Overview**, then fix your demo until it passes.
2. Implement a failing test/fixture for **Trace Format**, then fix your demo until it passes.
3. Implement a failing test/fixture for **Implementing the Loop**, then fix your demo until it passes.
4. Implement a failing test/fixture for **Native Tool Calling vs Text ReAct**, then fix your demo until it passes.
5. Implement a failing test/fixture for **Errors & Recovery**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
